In [1]:
import pandas as pd

def crear_variable(sub, ses, run, seg):
    sub_str, ses_str, run_str, seg_str = f"{sub:03d}", f"{ses:02d}", f"{run:02d}", f"{seg:02d}"
    json = f"DATOS/02-derived-ecg-dataset/datos/sub-{sub_str}/ses-{ses_str}/run-{run_str}/sub-{sub_str}_ses-{ses_str}_task-szMonitoring_run-{run_str}_seg-{seg_str}.json"
    npz  = f"DATOS/02-derived-ecg-dataset/datos/sub-{sub_str}/ses-{ses_str}/run-{run_str}/sub-{sub_str}_ses-{ses_str}_task-szMonitoring_run-{run_str}_seg-{seg_str}.npz"

    return npz, json, (sub,ses,run,seg)

In [2]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from scipy.stats import entropy
import json

def procesamiento(datos, win_size=30, step=15, smooth_n=5, overview_ds=10):
    """
    datos: (npz_path, json_path)
    Devuelve un DataFrame con métricas por ventanas deslizantes y, en la fila 0,
    RAW_t (tiempo absoluto decimado) y RAW_ecg (señal procesada decimada).
    """
    # =======================
    # Parámetros
    # =======================
    WIN_SIZE = float(win_size)                 # segundos
    STEP = float(step)                         # segundos
    SMOOTH_N = int(smooth_n)                   # puntos de suavizado (media móvil)
    DOWNSAMPLE_OVERVIEW = int(overview_ds)     # factor de decimación para overview

    npz_path, json_path, info = datos

    # =======================
    # Cargar metadata (JSON)
    # =======================
    with open(json_path, "r") as f:
        meta = json.load(f)
    fs = float(meta["fs_hz"])
    onset_abs = float(meta.get("onset_sec", 0.0))
    offset_abs = onset_abs + float(meta.get("duration_sec", 0.0))
    t_cut0 = float(meta.get("cut_start_sec", 0.0))  # inicio absoluto del segmento

    # =======================
    # Cargar ECG desde NPZ
    # =======================
    npz = np.load(npz_path)
    # Detectar vector 1D largo como ECG
    ecg = None
    for k in npz.files:
        arr = np.asarray(npz[k])
        if arr.ndim == 1 and arr.size > 100:
            ecg = arr.astype(float)
            break
    if ecg is None:
        raise RuntimeError(f"No encontré señal 1D en {npz_path}. Claves: {npz.files}")

    # Eje de tiempo (absoluto). Si el npz trae 't'/'time', lo uso y lo paso a absoluto.
    t = None
    for k in npz.files:
        if k.lower() in ("t", "time", "times"):
            t = np.asarray(npz[k], dtype=float)
            break
    if t is None:
        t_abs = np.arange(len(ecg))/fs + t_cut0
    else:
        # si es relativo (arranca en 0), igual desplazar no rompe si ya es absoluto
        t_abs = t + t_cut0

    dur_total = len(ecg) / fs
    print(f"fs={fs:.1f} Hz | duración={dur_total:.1f} s | muestras={len(ecg):,}")
    print(f"onset={onset_abs:.2f}s | offset={offset_abs:.2f}s | cut_start={t_cut0:.2f}s")

    # =======================
    # Preprocesamiento básico ECG
    # (igual que tu flujo: clean con NeuroKit; añade filtro/nota si necesitas)
    # =======================
    ecg_clean = nk.ecg_clean(ecg, sampling_rate=fs, method="neurokit")

    # =======================
    # Ventanas deslizantes (en segundos)
    # =======================
    def make_segments(signal, fs, win_s=30.0, step_s=15.0):
        dur = len(signal) / fs
        segments = []
        starts = np.arange(0, max(dur - win_s, 0) + 1e-9, step_s)
        for start in starts:
            s = int(start * fs); e = int((start + win_s) * fs)
            if e <= len(signal):
                segments.append((start, start + win_s, signal[s:e]))
        return segments

    segments = make_segments(ecg_clean, fs, WIN_SIZE, STEP)
    print(f"Total de ventanas: {len(segments)} (win={WIN_SIZE:.0f}s, step={STEP:.0f}s)")

    # =======================
    # Helpers
    # =======================
    def rr_entropy(rr_ms, bins=30):
        rr_ms = np.asarray(rr_ms, dtype=float)
        if rr_ms.size < 3 or not np.all(np.isfinite(rr_ms)):
            return np.nan
        hist, _ = np.histogram(rr_ms, bins=bins, density=True)
        return float(entropy(hist + 1e-12))

    def moving_mean_nanaware(y, n: int):
        """Media móvil nan-safe (n puntos)."""
        n = max(int(n), 1)
        y = np.asarray(y, dtype=float)
        if y.size == 0 or n == 1:
            return y
        w = np.ones(n, dtype=float)
        y_filled = np.nan_to_num(y, nan=0.0)
        valid = np.isfinite(y).astype(float)
        num = np.convolve(y_filled, w, mode="same")
        den = np.convolve(valid,   w, mode="same")
        out = num / den
        out[den == 0] = np.nan
        return out

    # =======================
    # Extraer features por ventana
    # =======================
    rows = []
    for (start, end, segment) in segments:
        # Detección de R
        signals, info = nk.ecg_peaks(segment, sampling_rate=fs)
        rpeaks = info.get("ECG_R_Peaks", None)

        if rpeaks is None or len(rpeaks) < 3:
            rows.append({
                "start_s": start + t_cut0,  # tiempo absoluto del inicio de ventana
                "end_s":   end   + t_cut0,  # tiempo absoluto del fin de ventana
                "HR_mean": np.nan, "SDNN_ms": np.nan, "RMSSD_ms": np.nan,
                "HF": np.nan, "RR_entropy": np.nan, "n_beats": 0
            })
            continue

        # HR promedio (bpm) como en tu flujo original (desea longitud del segmento)
        hr_sig = nk.ecg_rate(rpeaks, sampling_rate=fs, desired_length=len(segment))
        hr_mean = float(np.nanmean(hr_sig))

        # HRV tiempo (usar el dict 'info' completo)
        hrv_t = nk.hrv_time(info, sampling_rate=fs, show=False)
        SDNN  = float(hrv_t.get("HRV_SDNN",  pd.Series([np.nan])).iloc[0])
        RMSSD = float(hrv_t.get("HRV_RMSSD", pd.Series([np.nan])).iloc[0])

        # Entropía de RR (ms)
        rr_samples = np.diff(rpeaks)
        rr_ms = rr_samples / fs * 1000.0
        rr_ent = rr_entropy(rr_ms, bins=30)

        # HRV frecuencia (HF): pasar también el dict 'info'
        HF = np.nan
        try:
            hrv_f = nk.hrv_frequency(info, sampling_rate=fs, show=False,
                                     psd_method="welch", interpolation_rate=4)
        except Exception:
            try:
                hrv_f = nk.hrv_frequency(info, sampling_rate=fs, show=False,
                                         psd_method="lomb", interpolation_rate=4)
            except Exception:
                hrv_f = None

        if hrv_f is not None and not hrv_f.empty:
            HF = float(hrv_f.get("HRV_HF", pd.Series([np.nan])).iloc[0])

        rows.append({
            "start_s": start + t_cut0,
            "end_s":   end   + t_cut0,
            "HR_mean": hr_mean, "SDNN_ms": SDNN, "RMSSD_ms": RMSSD,
            "HF": HF, "RR_entropy": rr_ent, "n_beats": int(len(rpeaks))
        })

    df = pd.DataFrame(rows).sort_values("start_s").reset_index(drop=True)

    # Convertir a float para evitar dtypes 'object'
    for col in ["HR_mean","SDNN_ms","RMSSD_ms","HF","RR_entropy"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Suavizado opcional
    if SMOOTH_N and SMOOTH_N > 1 and not df.empty:
        for col in ["HR_mean","SDNN_ms","RMSSD_ms","HF","RR_entropy"]:
            df[col] = moving_mean_nanaware(df[col].values, SMOOTH_N)

    # =======================
    # Guardar señal decimada para "overview" (fila 0)
    # =======================
    step_ds = max(DOWNSAMPLE_OVERVIEW, 1)
    t_full_abs = t_abs
    x_full = ecg_clean
    t_ds = t_full_abs[::step_ds]
    x_ds = x_full[::step_ds]

    # crea columnas con dtype=object para poder guardar arrays
    df["RAW_t"]   = pd.Series([None]*len(df), dtype=object)
    df["RAW_ecg"] = pd.Series([None]*len(df), dtype=object)
    if len(df) > 0:
        df.at[0, "RAW_t"]   = t_ds
        df.at[0, "RAW_ecg"] = x_ds

    # (opcional) anexa onset/offset absolutos para tu viewer
    df.attrs["onset_abs"]  = onset_abs
    df.attrs["offset_abs"] = offset_abs
    df.attrs["fs"]         = fs
    df.attrs["npz_path"]   = npz_path
    df.attrs["json_path"]  = json_path
    
    # Guarda metadatos para usar después
    df.attrs["sub"] = datos[2][0]    # ej: "001"
    df.attrs["ses"] = datos[2][1]    # ej: "01"
    df.attrs["run"] = datos[2][2]    # ej: "03"
    df.attrs["seg"] = datos[2][3] 

    return df


In [3]:
import numpy as np
import plotly.graph_objs as go

def _smooth_series(y, n):
    n = max(int(n), 1)
    y = np.asarray(y, dtype=float)
    if n == 1 or not np.isfinite(y).any():
        return y
    y_nan = np.where(np.isfinite(y), y, 0.0)
    valid = np.where(np.isfinite(y), 1.0, 0.0)
    kernel = np.ones(n, dtype=float)
    num = np.convolve(y_nan, kernel, mode="same")
    den = np.convolve(valid, kernel, mode="same")
    out = np.full_like(num, np.nan, dtype=float)
    mask = den > 0
    out[mask] = num[mask] / den[mask]
    return out

def viewer_plotly_params_from_df(
    df,
    events=None,
    metrics=("HR_mean","SDNN_ms","RMSSD_ms","HF","RR_entropy"),
    smooth_options=(1,3,5,9),
    default_metric="HF",
    default_smooth=5,
    open_in_browser=False
):
    """
    Visualiza:
      - métricas por ventana (columnas en df)
      - señal procesada downsampleada si df tiene 'RAW_t' y 'RAW_ecg' (guardadas en df.loc[0, ...])

    También dibuja onset/offset:
      - Si 'events' es None, intenta df.attrs['onset_abs'] y df.attrs['offset_abs'].
      - Si 'events' es una lista de dicts con 'onset_sec' y 'duration_sec', usa esos.
    """
    # --------- detectar señal raw ---------
    has_raw = (
        "RAW_t" in df.columns and "RAW_ecg" in df.columns
        and len(df) > 0 and (df.loc[0, "RAW_t"] is not None)
    )

    # --------- filtrar métricas que existen ---------
    metrics = [m for m in metrics if m in df.columns]
    if not metrics and not has_raw:
        raise ValueError("df no contiene métricas ni señal cruda.")

    metrics_all = metrics.copy()
    RAW_KEY = None
    if has_raw:
        RAW_KEY = "ECG_raw"
        metrics_all.append(RAW_KEY)

    # --------- defaults robustos ---------
    if default_metric not in metrics_all:
        default_metric = RAW_KEY if RAW_KEY is not None else metrics_all[0]
    smooth_options = tuple(sorted(set(int(s) for s in smooth_options if int(s) >= 1)))
    if not smooth_options:
        smooth_options = (1,)
    if default_smooth not in smooth_options:
        default_smooth = smooth_options[0]

    # --------- eje X de métricas (usar inicio de ventana en absoluto) ---------
    if "start_s" in df.columns:
        x_win = np.asarray(df["start_s"].values, dtype=float)
    else:
        x_win = np.arange(len(df), dtype=float)

    # --------- precompute curvas ---------
    data = {}
    # métricas
    for m in metrics:
        y = np.asarray(df[m].values, dtype=float)
        for N in smooth_options:
            data[(m, N)] = (x_win, y, _smooth_series(y, N))
    # raw (overview)
    if has_raw:
        t_raw = np.asarray(df.loc[0, "RAW_t"], dtype=float)
        y_raw = np.asarray(df.loc[0, "RAW_ecg"], dtype=float)
        for N in smooth_options:
            data[(RAW_KEY, N)] = (t_raw, y_raw, _smooth_series(y_raw, N))

    # --------- construir figura ---------
    fig = go.Figure()
    visibility = {}
    k = 0
    for m in metrics_all:
        for N in smooth_options:
            x_vec, y_raw, y_sm = data[(m, N)]
            vis = (m == default_metric and N == default_smooth)
            fig.add_trace(go.Scatter(x=x_vec, y=y_raw, mode="lines",
                                     name=f"{m} raw", opacity=0.35, visible=vis))
            fig.add_trace(go.Scatter(x=x_vec, y=y_sm,  mode="lines",
                                     name=f"{m} smooth (N={N})", visible=vis))
            visibility[(m, N)] = (k, k+1)
            k += 2

    # --------- onset/offset (events o attrs del df) ---------
    shape_lines = []
    shape_rects = []

    # Si no pasan events, intenta desde los attrs del df
    if not events:
        onset = df.attrs.get("onset_abs", None)
        offset = df.attrs.get("offset_abs", None)
        if onset is not None and offset is not None:
            events = [{"onset_sec": float(onset), "duration_sec": float(offset - onset)}]

    if events:
        # Para sombrear usamos el eje y actual de la figura (paper para altura completa)
        for ev in events:
            on = ev.get("onset_sec", None)
            dur = ev.get("duration_sec", None)
            off = on + dur if (on is not None and dur is not None) else None
            if on is not None:
                shape_lines.append(
                    dict(type="line", x0=on, x1=on, y0=0, y1=1,
                         xref="x", yref="paper",
                         line=dict(color="red", dash="dash"))
                )
            if off is not None:
                shape_lines.append(
                    dict(type="line", x0=off, x1=off, y0=0, y1=1,
                         xref="x", yref="paper",
                         line=dict(color="orange", dash="dash"))
                )
                # sombreado
                shape_rects.append(
                    dict(type="rect", x0=on, x1=off, y0=0, y1=1,
                         xref="x", yref="paper",
                         opacity=0.12, line=dict(width=0), fillcolor="gray")
                )

    sub  = df.attrs.get("sub", "???")
    ses  = df.attrs.get("ses", "??")
    run  = df.attrs.get("run", "??")
    seg  = df.attrs.get("seg", "??")

    base_title = f"Crisis | Sub {sub} Ses {ses} Run {run} Seg {seg}"
    
    fig.update_layout(
        title=dict(
            text=f"{base_title} <br>Métrica: {default_metric}",
            x=0.5,
            xanchor="center",
            yanchor="top",
            font=dict(size=20)
        ),
        margin=dict(t=120),  # espacio superior amplio
        xaxis_title="Tiempo (s, absoluto)",
        yaxis_title=f"{default_metric}",
        shapes=(shape_lines + shape_rects),
        xaxis=dict(
            rangeslider=dict(visible=True),
            type="linear"
        ),
        legend=dict(orientation="h")
    )

    # --------- menús (ver nota en el mensaje) ---------
    total_traces = 2 * len(metrics_all) * len(smooth_options)

    # 1) Selector de métrica -> muestra (m, default_smooth)
    metric_buttons = []
    for m in metrics_all:
        vis_array = [False] * total_traces
        i_raw, i_sm = visibility[(m, default_smooth)]
        vis_array[i_raw] = True
        vis_array[i_sm]  = True
        metric_buttons.append(dict(
            label=m,
            method="update",
            args=[
            {"visible": vis_array},
            {
                "title": {"text": f"{base_title} <br>Métrica: {m}",
                            "x": 0.5,
                            "xanchor": "center",
                            "yanchor": "top",
                            "font": {"size": 20}},
                "yaxis": {"title": m}
            }
        ]
    ))
    # 2) Selector de suavizado -> muestra (default_metric, N)
    smooth_buttons = []
    for N in smooth_options:
        vis_array = [False] * total_traces
        i_raw, i_sm = visibility[(default_metric, N)]
        vis_array[i_raw] = True
        vis_array[i_sm]  = True
        smooth_buttons.append(dict(
            label=f"N={N}",
            method="update",
            args=[
                    {"visible": vis_array},
                    {
                        "title": {
                            "text": f"{base_title} <br>Métrica: {default_metric} (N={N})",
                            "x": 0.5,
                            "xanchor": "center",
                            "yanchor": "top",
                            "font": {"size": 20}
                        }
                    }
                ]
    ))

    fig.update_layout(
        updatemenus=[
            dict(type="dropdown", direction="down", x=0.0,  y=1.18,
                 buttons=metric_buttons, showactive=True, xanchor="left"),
            dict(type="dropdown", direction="down", x=0.25, y=1.18,
                 buttons=smooth_buttons, showactive=True, xanchor="left")
        ],
        annotations=[
            dict(text="Métrica:", x=0.0,  y=1.25, xref="paper", yref="paper", showarrow=False),
            dict(text="Smooth:",  x=0.25, y=1.25, xref="paper", yref="paper", showarrow=False),
        ]
    )

    if open_in_browser:
        fig.show(renderer="browser")
    else:
        fig.show()


In [4]:
def ecg(sub, ses, run, seg):
    datos = crear_variable(sub, ses, run, seg)
    df_ecg = procesamiento(datos, win_size=60, step=15, smooth_n=5, overview_ds=10)
    viewer_plotly_params_from_df(
        df_ecg,
        events=None,
        metrics=("HR_mean","SDNN_ms","RMSSD_ms","HF","RR_entropy"),
        smooth_options=(1,3,5,9),
        default_metric="HF",
        default_smooth=5,
        open_in_browser=True
    )

In [5]:
ecg(1, 1, 3, 0)

fs=256.0 Hz | duración=1872.0 s | muestras=479,232
onset=57975.00s | offset=58047.00s | cut_start=57075.00s
Total de ventanas: 121 (win=60s, step=15s)
